# CMIP6 metadata and clisops subsetting

Compare synthetic NetCDF controls and metadata variants using the public clisops subset operator. Each case checks xarray decoding first, then runs independent time-only and geographic bounding-box requests. The checks compare the returned coordinates and data values with explicit selections.

In [1]:
import sys
import warnings
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import xarray as xr
from clisops.ops.subset import subset
from IPython.display import display
from netCDF4 import Dataset

print(f"Python {sys.version.split()[0]}")
for package in ("clisops", "xarray", "cf-xarray", "numpy", "pandas", "netCDF4", "cftime"):
    print(f"{package} {version(package)}")

Python 3.13.18
clisops 0.18.1
xarray 2025.10.1
cf-xarray 0.11.3
numpy 2.5.4
pandas 3.1.1
netCDF4 1.7.5
cftime 1.6.5


## Shared fixtures and checks

Use four daily samples from 1 to 4 January 1850 and request the middle two days. The bbox is 12°W to 3°W and 43°N to 52°N. On the rectilinear grid it includes latitudes 45° and 50° and longitudes 10°W and 5°W. Unique data values make a shifted selection visible.

In [2]:
TIME_UNITS = "days since 1850-01-01 00:00:00"
DATES = np.arange("1850-01-01", "1850-01-05", dtype="datetime64[D]").astype("datetime64[ns]")
TIME_RANGE = "1850-01-02T00:00:00/1850-01-03T00:00:00"
BBOX = (-12, 43, -3, 52)  # west, south, east, north

time = (
    "time",
    np.arange(4),
    {"units": TIME_UNITS, "calendar": "standard", "axis": "T", "standard_name": "time"},
)
latitude = (
    "lat",
    [40, 45, 50, 55],
    {"units": "degrees_north", "axis": "Y", "standard_name": "latitude"},
)
longitude = (
    "lon",
    [-15, -10, -5, 0, 5],
    {"units": "degrees_east", "axis": "X", "standard_name": "longitude"},
)
pressure = (
    "plev",
    [100000, 85000],
    {"units": "Pa", "axis": "Z", "standard_name": "air_pressure", "positive": "down"},
)

rectilinear = xr.Dataset(
    {"ta": (("time", "plev", "lat", "lon"), np.arange(160.0).reshape(4, 2, 4, 5))},
    coords={"time": time, "plev": pressure, "lat": latitude, "lon": longitude},
)
rectilinear_bbox = {"lat": slice(1, 3), "lon": slice(1, 3)}

In [3]:
results = []


def check(case, stage, action):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            value = action()
        except Exception as exc:
            value = None
            error = f"{type(exc).__name__}: {exc}"
        else:
            error = None

    status = "FAIL" if error else "PASS (warning)" if caught else "PASS"
    results.append({
        "case": case,
        "stage": stage,
        "status": status,
        "error": error,
        "warnings": [f"{w.category.__name__}: {w.message}" for w in caught],
    })
    return value


def run_case(case, raw, variable, bbox_index, bbox_mask=None):
    # Use positional expectations, independently of clisops' coordinate discovery.
    expected = raw[variable].assign_coords(time=DATES)
    expected_bbox = expected.isel(bbox_index)
    if bbox_mask is not None:
        expected_bbox = expected_bbox.where(bbox_mask)

    with TemporaryDirectory() as directory:
        path = Path(directory) / "fixture.nc"
        raw.to_netcdf(path, engine="netcdf4")

        # Confirm that xarray wrote the exact strings the decoder should see.
        with Dataset(path) as file:
            assert file["time"].units == raw["time"].attrs["units"]
            if "coordinates" in raw[variable].attrs:
                assert file[variable].coordinates == raw[variable].attrs["coordinates"]
                assert ("height" in file.variables) == ("height" in raw.coords)

        def open_fixture():
            with xr.open_dataset(path, engine="netcdf4") as decoded:
                actual = decoded.load()
            np.testing.assert_array_equal(actual.time.values, DATES)
            xr.testing.assert_equal(actual[variable], expected)
            return actual

        decoded = check(case, "xarray open", open_fixture)

    requests = [
        ("time subset", {"time": TIME_RANGE}, expected.isel(time=slice(1, 3))),
        ("bbox subset", {"area": BBOX}, expected_bbox),
    ]
    for stage, parameters, expected_subset in requests:
        if decoded is None:
            results.append({
                "case": case, "stage": stage, "status": "SKIP", "error": None, "warnings": [],
            })
            continue

        def verify_subset():
            (actual,) = subset(decoded.copy(deep=True), output_type="xarray", **parameters)
            xr.testing.assert_equal(actual[variable], expected_subset)

        check(case, stage, verify_subset)

## 1. Time units

Keep the numeric offsets and calendar fixed. Change only the formatting of the reference time in the units string.

In [4]:
run_case("Rectilinear control", rectilinear, "ta", rectilinear_bbox)

short_time_units = rectilinear.copy(deep=True)
short_time_units.time.attrs["units"] = "days since 1850-01-01 0:0:0.0"
run_case("Unpadded time units", short_time_units, "ta", rectilinear_bbox)

## 2. Missing coordinate metadata

Use the rectilinear control from case 1. Remove one attribute from one coordinate at a time. Both pressure levels should be preserved by each request.

In [5]:
for coordinate in ("time", "lat", "lon", "plev"):
    for attribute in ("axis", "standard_name"):
        missing_attribute = rectilinear.copy(deep=True)
        del missing_attribute[coordinate].attrs[attribute]
        run_case(f"{coordinate} without {attribute}", missing_attribute, "ta", rectilinear_bbox)

## 3. Degree-valued projected axes

Keep the 2-D geographic coordinates identical in both fixtures. Replace the metre-valued x/y axes with small degree-valued axes, retaining their projected standard names. The geographic bbox retains three cells in a 2 × 2 window. The northeast cell is outside the bbox and should be masked.

In [6]:
curvilinear = xr.Dataset(
    {"tas": (("time", "y", "x"), np.arange(48.0).reshape(4, 3, 4))},
    coords={
        "time": time,
        "x": ("x", [0, 500_000, 1_000_000, 1_500_000],
              {"standard_name": "projection_x_coordinate", "units": "m", "axis": "X"}),
        "y": ("y", [0, 500_000, 1_000_000],
              {"standard_name": "projection_y_coordinate", "units": "m", "axis": "Y"}),
        "lat": (("y", "x"), [[39, 40, 41, 42], [43, 45, 46, 47], [49, 50, 53, 54]],
                {"standard_name": "latitude", "units": "degrees_north"}),
        "lon": (("y", "x"), [[-16, -11, -6, -1], [-15, -10, -5, 0], [-14, -9, -4, 1]],
                {"standard_name": "longitude", "units": "degrees_east"}),
    },
)
curvilinear_bbox = {"y": slice(1, 3), "x": slice(1, 3)}
curvilinear_mask = xr.DataArray([[True, True], [True, False]], dims=("y", "x"))

run_case("Projected axes in metres", curvilinear, "tas", curvilinear_bbox, curvilinear_mask)

axes_in_degrees = curvilinear.assign_coords(
    x=("x", [0, 5, 10, 15], {**curvilinear.x.attrs, "units": "degrees"}),
    y=("y", [0, 5, 10], {**curvilinear.y.attrs, "units": "degrees"}),
)
run_case("Projected axes in degrees", axes_in_degrees, "tas", curvilinear_bbox, curvilinear_mask)

## 4. Missing auxiliary height coordinate

The control has a scalar 2 m height. Both files declare `coordinates="height"` on `tas`. The variant omits the height variable.

In [7]:
height_control = xr.Dataset(
    {"tas": (("time", "lat", "lon"), np.arange(80.0).reshape(4, 4, 5), {"coordinates": "height"})},
    coords={
        "time": time,
        "lat": latitude,
        "lon": longitude,
        "height": ((), 2.0, {"standard_name": "height", "units": "m", "positive": "up"}),
    },
)
run_case("Height control", height_control, "tas", rectilinear_bbox)
run_case("Missing height", height_control.drop_vars("height"), "tas", rectilinear_bbox)

## Results

A pass means the coordinates and values matched the expected selection. If xarray cannot open and verify a fixture, both clisops requests are skipped. The results apply to the synthetic fixtures and the versions printed above.

In [8]:
summary = pd.DataFrame(results)
stages = ["xarray open", "time subset", "bbox subset"]
display(summary.pivot(index="case", columns="stage", values="status")
        .reindex(index=summary["case"].unique(), columns=stages))


warnings_by_stage = {}
for result in results:
    case, stage = result["case"], result["stage"]
    if result["error"]:
        print(f'{case} / {stage}: {result["error"]}')
    for warning in result["warnings"]:
        warnings_by_stage.setdefault((stage, warning), []).append(case)

for (stage, warning), cases in warnings_by_stage.items():
    scope = "all fixtures" if len(cases) == summary["case"].nunique() else ", ".join(cases)
    print(f"{stage} ({scope}): {warning}")

stage,xarray open,time subset,bbox subset
case,,,
Rectilinear control,PASS,PASS,PASS (warning)
Unpadded time units,PASS,PASS,PASS (warning)
time without axis,PASS,PASS,PASS (warning)
time without standard_name,PASS,PASS,PASS (warning)
lat without axis,PASS,PASS,PASS (warning)
lat without standard_name,PASS,PASS,PASS (warning)
lon without axis,PASS,PASS,PASS (warning)
lon without standard_name,PASS,PASS,PASS (warning)
plev without axis,PASS,PASS,PASS (warning)


bbox subset (all fixtures): UserWarning: For coordinate variable 'lon' no bounds can be identified.
